# ISP - EOT Submission

## Task 1.2 — Downtown Car Counter

### Overview
This cell defines and runs the `analyseVideo` function, which counts the number of cars
heading toward downtown using **background subtraction**, **centroid tracking**, and
**direction filtering**.

### How Pipeline Works?
1. **Open video** — loads the video file and validates it
2. **Gaussian Blur** — smooths the frame to reduce noise before detection
3. **KNN Background Subtraction** — separates moving foreground (cars) from static background
4. **Thresholding** — converts the mask to clean black & white
5. **Morphological Closing** — fills small holes and gaps in detected car shapes
6. **Contour Detection** — finds outlines of detected objects
7. **Area Filtering** — removes objects too small or too large to be vehicles
8. **Centroid Tracking** — matches each detection to an existing tracked car using distance
9. **Position History** — stores last 5 positions to compute average movement direction
10. **Direction Filtering** — only counts cars moving LEFT (toward downtown), blocks right-moving cars
11. **Counting Zone** — only cars passing through the defined red box are eligible to be counted

### Key Parameters
- `history = 500` — number of past frames used to model the background
- `dist2Threshold = 400` — sensitivity of KNN detection (lower = more sensitive)
- `MIN_AREA = 3000` — minimum contour area to be considered a vehicle
- `MAX_AREA = 50000` — maximum contour area to filter out noise/merged blobs
- `MAX_MATCH_DIST = 200` — maximum pixel distance to match a detection to a tracked car
- `avg_dx < 1` — direction threshold
- `LEFT/RIGHT = 840, 1010` — horizontal boundaries of the downtown counting zone
- `TOP/BOTTOM = 200, 345` — vertical boundaries of the downtown counting zone

### Box Color Guide
- **Green box** — car detected inside the counting zone
- **Blue box** — car detected outside the counting zone
- **Red box** — downtown counting zone boundary

### Video Controls
- Press **`q`** to stop the video at any time


## Install Necessary Packages

In [1]:
!pip install numpy
!pip install opencv-python

## Import Libraries

In [2]:
import numpy as np
import cv2
import os
base_directory = os.getcwd()
print(base_directory)

C:\Users\Aswat\Exercise_1_ISP_Final


## Tracking Downtown Moving Cars

In [3]:
class Car:

    def __init__(self, id, position):
        # Unique ID for this car
        self.id = id
        # Current position of the car's center
        self.position = position
        # Previous position used to compute movement direction
        self.prev_position = position
        # Flag to ensure each car is counted only once
        self.counted = False
        # Number of consecutive frames the car was not detected
        self.frames_missing = 0
        # Number of frames this car has been actively tracked
        self.frames_tracked = 0
        # Last 5 positions used to compute average movement direction
        self.position_history = [position]


def is_inside_zone(center, roi_polygon):
    return cv2.pointPolygonTest(roi_polygon, (float(center[0]), float(center[1])), False) >= 0


def distance(p1, p2):
    return np.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)


def analyseVideo(video_path):

    # Open video file
    video = cv2.VideoCapture(video_path)
    if not video.isOpened():
        print(f"Error: Could not open video: {video_path}")
        return 0, 0

    # Get video properties
    fps          = video.get(cv2.CAP_PROP_FPS)
    video_width  = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
    video_height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
    duration_sec = total_frames / fps

    print(f"Video: {video_path.split(os.sep)[-1]}")
    print(f"Size : {video_width}x{video_height} | FPS:{fps:.1f} | Duration:{duration_sec:.1f}s")

    # KNN background subtractor which separates moving cars from static background
    # history: number of past frames to build background model
    # dist2Threshold: sensitivity of foreground detection
    # detectShadows: disabled to avoid shadow pixels being treated as cars
    background_remover = cv2.createBackgroundSubtractorKNN(
        detectShadows=False,
        history=500,
        dist2Threshold=400
    )

    # Kernel for morphological closing to fill gaps in detected shapes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

    # Downtown counting zone whereby only cars passing through this box are counted
    LEFT, TOP, RIGHT, BOTTOM = 840, 200, 1010, 345
    center_x   = (LEFT + RIGHT)  / 2.0
    center_y   = (TOP  + BOTTOM) / 2.0
    roi_width  = float(RIGHT - LEFT)
    roi_height = float(BOTTOM - TOP)

    # Counters and trackers
    vehicle_count = 0
    car_id        = 0
    frame_count   = 0
    cars          = {}

    while video.isOpened():
        ret, frame = video.read()
        if not ret:
            break

        frame_count += 1

        # Increment missing frame counter for all currently tracked cars
        for car in cars.values():
            car.frames_missing += 1

        # Apply Gaussian blur to reduce noise before background subtraction
        smoothed_frame  = cv2.GaussianBlur(frame, (5, 5), 0)

        # Apply KNN background subtraction to get foreground mask
        foreground_mask = background_remover.apply(smoothed_frame)

        # Convert mask to strict black & white — removes grey shadow pixels
        _, foreground_mask = cv2.threshold(foreground_mask, 100, 255, cv2.THRESH_BINARY)

        # Morphological closing that fills small holes and gaps in detected car shapes
        foreground_mask = cv2.morphologyEx(foreground_mask, cv2.MORPH_CLOSE, kernel, iterations=2)

        # Find external contours of all white regions in the mask
        contours, _ = cv2.findContours(foreground_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        # Build the downtown counting zone polygon
        rect        = ((center_x, center_y), (roi_width, roi_height), 0)
        roi_polygon = cv2.boxPoints(rect).astype(np.int32)

        detected_in_zone = 0

        for contour in contours:
            x, y, w, h = cv2.boundingRect(contour)
            area        = cv2.contourArea(contour)

            # Filter out contours too small or too large to be vehicles
            if area < 3000 or area > 50000:
                continue

            # Compute the center of this detection
            center = (x + w // 2, y + h // 2)

            # Find closest existing car within matching distance
            matched_car = None
            min_dist    = 200

            for car in cars.values():
                dist = distance(center, car.position)
                if dist < min_dist:
                    min_dist    = dist
                    matched_car = car

            if matched_car is not None:
                # Update existing car's position and tracking history
                matched_car.prev_position  = matched_car.position
                matched_car.position       = center
                matched_car.frames_missing = 0
                matched_car.frames_tracked += 1
                matched_car.position_history.append(center)
                # Keep only last 5 positions for direction calculation
                if len(matched_car.position_history) > 5:
                    matched_car.position_history.pop(0)
            else:
                # If no match found then register as a new car
                cars[car_id] = Car(car_id, center)
                matched_car  = cars[car_id]
                car_id      += 1

            # Check if car is inside the downtown counting zone
            if is_inside_zone(center, roi_polygon):
                detected_in_zone += 1

                if not matched_car.counted:
                    # Need at least 3 positions to compute a reliable direction
                    if len(matched_car.position_history) >= 3:
                        # Compute average horizontal displacement across tracked positions
                        avg_dx = (
                            matched_car.position_history[-1][0] -
                            matched_car.position_history[0][0]
                        ) / len(matched_car.position_history)

                        # Downtown cars if avg_dx < 1 else avg_dx >= 1 are blocked
                        if avg_dx < 1:
                            vehicle_count      += 1
                            matched_car.counted = True
                            print(f"Car #{vehicle_count} to downtown | ID:{matched_car.id} | Frame:{frame_count} | Pos:{center} | avg_dx:{avg_dx:.2f}")

                # Green box for cars inside the zone
                cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 3)
                cv2.putText(frame, f"{int(area)}", (x, y - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            else:
                # Blue box for cars outside the zone
                cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)

        # Remove cars that have been missing for more than 10 frames
        for cid in [cid for cid, c in cars.items() if c.frames_missing > 10]:
            del cars[cid]

        # Draw the downtown counting zone boundary in red
        cv2.drawContours(frame, [roi_polygon], 0, (0, 0, 255), 4)
        cv2.putText(frame, "TO DOWNTOWN",
                    (int(center_x) - 70, int(center_y) - int(roi_height / 2) - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # Live display running stats on the frame
        elapsed_min = (frame_count / fps) / 60
        cpm         = vehicle_count / elapsed_min if elapsed_min > 0 else 0
        cv2.putText(frame, f"Downtown Cars: {vehicle_count}",
                    (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        cv2.putText(frame, f"Cars/min: {cpm:.2f}",
                    (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        cv2.putText(frame, f"Frame: {frame_count} | In zone: {detected_in_zone}",
                    (10, 95), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        cv2.imshow("Downtown Counter", frame)
        if cv2.waitKey(10) == ord("q"):
            break

    video.release()
    cv2.destroyAllWindows()

    # Final results
    cpm_final = vehicle_count / (duration_sec / 60) if duration_sec > 0 else 0
    print("\n" + "=" * 50)
    print(f"TOTAL CARS TO DOWNTOWN: {vehicle_count}")
    print(f"CARS PER MINUTE       : {cpm_final:.2f}")
    print("=" * 50)
    return vehicle_count, cpm_final


# Run code section
video1 = os.path.join(base_directory, "Exercise1_Files", "Traffic_Laramie_1.mp4")
video2 = os.path.join(base_directory, "Exercise1_Files", "Traffic_Laramie_2.mp4")

print("### Traffic_Laramie_1.mp4 ###")
total1, cpm1 = analyseVideo(video1)

print("\n### Traffic_Laramie_2.mp4 ###")
total2, cpm2 = analyseVideo(video2)

print("\n" + "=" * 60)
print("TASK 1.2 RESULTS:")
print("=" * 60)
print(f"| {'Video':<23} | {'Total Cars':>10} | {'Cars/Minute':>11} |")
print(f"|-{'':-<23}-|-{'':-<10}-|-{'':-<11}-|")
print(f"| {'Traffic_Laramie_1.mp4':<23} | {total1:>10} | {cpm1:>11.2f} |")
print(f"| {'Traffic_Laramie_2.mp4':<23} | {total2:>10} | {cpm2:>11.2f} |")
print("=" * 60)


### Traffic_Laramie_1.mp4 ###
Video: Traffic_Laramie_1.mp4
Size : 1040x600 | FPS:25.0 | Duration:177.9s
Car #1 to downtown | ID:4 | Frame:334 | Pos:(1002, 337) | avg_dx:-1.33
Car #2 to downtown | ID:6 | Frame:726 | Pos:(915, 345) | avg_dx:-0.60
Car #3 to downtown | ID:2 | Frame:792 | Pos:(875, 345) | avg_dx:0.20
Car #4 to downtown | ID:7 | Frame:882 | Pos:(873, 343) | avg_dx:0.00
Car #5 to downtown | ID:9 | Frame:1130 | Pos:(879, 322) | avg_dx:-1.00
Car #6 to downtown | ID:17 | Frame:2339 | Pos:(994, 345) | avg_dx:-1.40
Car #7 to downtown | ID:21 | Frame:3121 | Pos:(874, 342) | avg_dx:0.60
Car #8 to downtown | ID:28 | Frame:3866 | Pos:(939, 344) | avg_dx:-2.40
Car #9 to downtown | ID:27 | Frame:4221 | Pos:(847, 341) | avg_dx:0.40
Car #10 to downtown | ID:30 | Frame:4362 | Pos:(915, 345) | avg_dx:0.00

TOTAL CARS TO DOWNTOWN: 10
CARS PER MINUTE       : 3.37

### Traffic_Laramie_2.mp4 ###
Video: Traffic_Laramie_2.mp4
Size : 1040x600 | FPS:25.0 | Duration:105.7s
Car #1 to downtown | ID:0 